#### Task 5: Two pitfalls when interpreting correlation, shown with simulated app data.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
 
rng = np.random.default_rng(seed=5)
 
# ---------- Pitfall 1: correlation is not causation (hidden confounder) - Zomato ----------
print("PITFALL 1: Correlation does not prove causation (confounding variable)")
days = 200
rain = rng.uniform(0, 50, days)                              # daily rainfall in mm (hidden third variable)
soup_orders = 300 + 6 * rain + rng.normal(0, 40, days)       # rain -> more soup orders
delivery_delay = 20 + 0.5 * rain + rng.normal(0, 4, days)    # rain -> slower deliveries
zom = pd.DataFrame({"rain": rain, "soup_orders": soup_orders, "delay": delivery_delay})
 
r_raw = zom["soup_orders"].corr(zom["delay"])
# Partial correlation: remove the effect of rain from both variables, then correlate residuals
res_soup = zom["soup_orders"] - np.polyval(np.polyfit(zom["rain"], zom["soup_orders"], 1), zom["rain"])
res_delay = zom["delay"] - np.polyval(np.polyfit(zom["rain"], zom["delay"], 1), zom["rain"])
r_partial = np.corrcoef(res_soup, res_delay)[0, 1]
print(f"  Correlation(soup orders, delivery delay)             = {r_raw:.2f}  (looks strong)")
print(f"  Same correlation after controlling for rainfall      = {r_partial:.2f}  (about zero)")
print("  Rain drives both, so soup orders do not cause the delays.\n")
 
# ---------- Pitfall 2: Pearson only measures LINEAR relationships (and is outlier-sensitive) - YouTube ----------
print("PITFALL 2: Pearson r misses non-linear patterns and is distorted by outliers")
length = np.linspace(1, 20, 60)                              # video length in minutes
engagement = -(length - 10.5) ** 2 + 100 + rng.normal(0, 4, 60)   # engagement peaks at about 10 minutes
r_len = np.corrcoef(length, engagement)[0, 1]
print(f"  Video length vs engagement (inverted U): Pearson r = {r_len:.2f}  (near zero)")
print("  Yet length clearly affects engagement: it rises up to about 10 minutes, then falls.\n")
 
views = rng.normal(10_000, 2_000, 15)
likes = rng.normal(500, 100, 15)
likes = likes - np.polyval(np.polyfit(views, likes, 1), views) + likes.mean()   # make likes unrelated to views
r_before = np.corrcoef(views, likes)[0, 1]
views_out, likes_out = np.append(views, 2_000_000), np.append(likes, 90_000)   # one viral video
r_after = np.corrcoef(views_out, likes_out)[0, 1]
print(f"  15 ordinary videos (views vs likes, unrelated):   r = {r_before:.2f}")
print(f"  Add ONE viral video (2M views, 90k likes):        r = {r_after:.2f}  (outlier creates a 'strong' link)")
rho_after = stats.spearmanr(views_out, likes_out)[0]
print(f"  Spearman rho with the same outlier included:     rho = {rho_after:.2f}  (more robust)")

PITFALL 1: Correlation does not prove causation (confounding variable)
  Correlation(soup orders, delivery delay)             = 0.79  (looks strong)
  Same correlation after controlling for rainfall      = -0.02  (about zero)
  Rain drives both, so soup orders do not cause the delays.

PITFALL 2: Pearson r misses non-linear patterns and is distorted by outliers
  Video length vs engagement (inverted U): Pearson r = 0.01  (near zero)
  Yet length clearly affects engagement: it rises up to about 10 minutes, then falls.

  15 ordinary videos (views vs likes, unrelated):   r = -0.00
  Add ONE viral video (2M views, 90k likes):        r = 1.00  (outlier creates a 'strong' link)
  Spearman rho with the same outlier included:     rho = 0.19  (more robust)
